# Sweeping a parameter

One `ExperimentDef` with a `scenario()` per value, not a dedicated "parameter
sweep" experiment type — the same shape as every other experiment here. This
notebook sweeps the credit pipeline's `approved.limit` across a handful of
values and plots how many records are approved at each one.

In [1]:
import polars as pl
from decider.experiments import Experiment, OutcomeCounts, RunStatus
from example_projects.experiments.credit_pipeline import build

ENTRY = "example_projects.experiments.credit_pipeline:build"
pipeline = build()
df = pl.DataFrame({
    "income": [1000.0, 2000.0, 1500.0, 800.0],
    "debt": [350.0, 900.0, 700.0, 300.0],
})
df

shape: (4, 2)
┌────────┬───────┐
│ income ┆ debt  │
│ ---    ┆ ---   │
│ f64    ┆ f64   │
╞════════╪═══════╡
│ 1000.0 ┆ 350.0 │
│ 2000.0 ┆ 900.0 │
│ 1500.0 ┆ 700.0 │
│ 800.0  ┆ 300.0 │
└────────┴───────┘


In [2]:
exp = Experiment("limit-sweep", pipeline, df, entry=ENTRY)
for limit in (0.2, 0.3, 0.4, 0.5, 0.6):
    exp.scenario(f"limit_{limit}", params={"credit": {"approved": {"limit": limit}}})
exp.plot(OutcomeCounts(type="outcome_counts", column="approved")).compare(outputs=("approved",))

result = exp.run()
assert result.status is RunStatus.COMPLETED
result.status

<RunStatus.COMPLETED: 'completed'>


Each scenario's `divergences` is how many records flipped relative to the
baseline — the pipeline run once with its own default params, not any one
scenario — so `limit_0.4` (the pipeline's actual default) shows zero: a cheap
way to see which sweep values actually changed anything.

In [3]:
[(s.name, len(s.divergences)) for s in result.scenarios]

[('limit_0.2', 0), ('limit_0.3', 0), ('limit_0.4', 1), ('limit_0.5', 4), ('limit_0.6', 4)]


The `OutcomeCounts` plot you attached ran once, over the baseline. A plot
that should run per scenario instead of once over the baseline is an open
question this prototype doesn't answer yet — see
`notes/experiment-plot-registry-spike.md`.

In [4]:
result.plots

{'outcome_counts': {'kind': 'bar', 'column': 'approved', 'series': {'False': 4}}}
